# Caso C · 01 EDA HVAC y catálogo de fallos

> _Tutorial · Caso de uso: **C — Anomalías HVAC** · Capa Medallion: **bronce** · Spec: `docs/specs/synthetic-bms/02-domain-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Conocer el dataset LBNL FDD (mock RTU) con 4 tipos de fallos etiquetados, identificar la firma de cada fallo en sensores y construir el catálogo del Caso C.


## 2. Qué se aprende

- 4 tipos de fallos HVAC y cómo se manifiestan.
- Variables: T_supply, T_return, valve, fan, occupancy.
- Conceptos ΔT, duty cycle, ratio fan/valve.
- Cómo separar fallos en clases supervisadas.


## 3. Contexto del caso de uso

Datos sintéticos del generador `caseC_faults.yaml` o LBNL FDD reducido. Las etiquetas viven en `captia_fault_labels` (measurement separado).


## 4. Relación con CENTINELA+

El sistema real puede sufrir estos 4 tipos. La descripción cualitativa fue solicitada a CAPTIA en el informe de mayo.


## 5. Relación con Medallion

Bronce mock LBNL FDD; etiquetas las usaremos para el supervised eval.


## 6. Datos de entrada

`notebooks/_data/lbnl_fdd_rtu_mock.csv`.


## 7. Schema CAPTIA esperado

Mapping LBNL → CAPTIA visto en docs:

| LBNL | CAPTIA | bucket |
|---|---|---|
| `SA_TEMP` | `temperature_supply` | telemetry |
| `RA_TEMP` | `temperature_return` | telemetry |
| `OA_TEMP` | `temperature_outdoor` | telemetry |
| `CCV` | `valve_control` | state_events |
| `FAN_STATE` | `fan_speed_01_state` | state_events |
| `OCCU_MOD` | `occupancy` | telemetry |
Etiquetas → `captia_fault_labels` (state_events).


## 8. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [ ]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


## 9. Carga de datos o mock

Cargamos el mock con cabecera explícita.


In [ ]:
csv_path = ROOT / "notebooks" / "_data" / "lbnl_fdd_rtu_mock.csv"
df = pd.read_csv(csv_path, comment="#", parse_dates=["timestamp"]).sort_values("timestamp")
df.head()


## 10. Exploración paso a paso

Distribución de los tipos de fallo.


In [ ]:
print(df["fault_type"].value_counts())


## 11. Transformación bronce → plata

(Lo veremos en el siguiente notebook.) Aquí calculamos features para EDA.


In [ ]:
df["dt_supply_return"] = df["RA_TEMP"] - df["SA_TEMP"]
df["dt_supply_outdoor"] = df["OA_TEMP"] - df["SA_TEMP"]
df["fan_eff"] = df["CCV"] - df["FAN_STATE"]  # idealmente 0; >0 = válvula abierta sin fan
df.head()


## 12. Construcción de capa oro

(Notebook 03).


## 13. Visualizaciones explicativas

T_supply durante valve_stuck (debería NO bajar pese a CCV=1).


In [ ]:
mask = df["fault_type"] == "valve_stuck"
if mask.any():
    win = df.loc[mask].head(120)
    plt.figure(figsize=(10, 3))
    plt.plot(win["timestamp"], win["SA_TEMP"], label="SA", color="#3F51B5")
    plt.plot(win["timestamp"], win["RA_TEMP"], label="RA", color="#FF5722")
    plt.plot(win["timestamp"], win["CCV"] * 5 + 18, label="valve x5", color="#4CAF50")
    plt.legend()
    plt.title("Valve stuck — T_supply no baja")
    plt.tight_layout()


## 14. Validaciones

Las etiquetas deben sumar al menos 5% del dataset (mocked).


In [ ]:
ratio = (df["is_fault"] == 1).mean()
print("Fault ratio:", ratio)
assert 0.05 <= ratio <= 0.6


## 15. Errores comunes

1. Asumir que el dataset solo tiene fallo→entonces el modelo no aprende lo normal.
2. Concatenar fallos sin solapamiento (el mock incluye solapamientos).
3. Mezclar `is_fault` y `fault_type` en el mismo modelo sin preprocesar.


## 16. Ejercicios propuestos

1. Cuenta cuántos episodios de cada tipo (no puntos).
2. Visualiza ΔT durante refrigerant_low.
3. Estima la duración media por tipo de fallo.


## 17. Cómo se reutiliza con datos reales

LBNL FDD real tiene mismo schema. Para CENTINELA+ los fallos vienen de tickets manuales — convertir a `captia_fault_labels`.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `03_case_C_hvac_anomaly_detection/02_bronze_to_silver_hvac.ipynb`.
- Documento web del caso: `docs/use-cases/case-c-hvac-anomaly.md`.
